# exp002 — ink_9um on PHerc 0841 / w00, correct depth input + metrics (Kaggle)

**Settings:** Accelerator **GPU T4 x2** (1 GPU used), Internet **on**, notebook **private**. Do not publish outputs.
Config: `configs/exp002_841w00_ink9um_s42.yaml`. Downloads: ~138 MB checkpoint, ~280 MB volume + ~10 MB labels
(into `/kaggle/working`, discarded with the session). Prints numbers only (no images).

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
python --version; df -h /kaggle/working | tail -1
git clone -q --depth 1 https://github.com/andreluizpedroso/Vesuvius-Challenge.git /kaggle/working/harness
git -C /kaggle/working/harness log -1 --format='harness %H'

In [ ]:
import os, yaml, pathlib
EXP = "configs/exp002_841w00_ink9um_s42.yaml"
cfg = yaml.safe_load((pathlib.Path("/kaggle/working/harness") / EXP).read_text())
os.environ.update({
    "EXP_NAME": cfg["experiment"],
    "EXP_CFG": f"/kaggle/working/harness/{EXP}",
    "VILLA_SHA": cfg["upstream"]["villa_commit"],
    "PY_VER": cfg["upstream"]["python"],
    "UV_SYNC_ARGS": " ".join(cfg["upstream"]["uv_sync_args"]),
    "CK_REPO": cfg["checkpoint"]["hf_repo"],
    "CK_REV": cfg["checkpoint"]["hf_revision"],
    "CK_FILE": cfg["checkpoint"]["file"],
    "INFER_MODULE": cfg["inference"]["module"],
    "INFER_ARGS": " ".join(cfg["inference"]["base_args"]),
    "OFFSETS": " ".join(str(k) for k in cfg["inference"]["depth_offsets"]),
    "VES": "/kaggle/working/villa/vesuvius",
    "CKPT": f"/kaggle/working/checkpoints/ink_9um/{cfg['checkpoint']['file']}",
    "DATA": "/kaggle/working/data/841_w00_exp002",
    "OUT": f"/kaggle/working/outputs/{cfg['experiment']}",
})
print({k: os.environ[k] for k in ["EXP_NAME", "VILLA_SHA", "CK_FILE", "OFFSETS"]})

## 1. Upstream code (villa, pinned commit) + environment

In [ ]:
%%bash
set -e
git init -q /kaggle/working/villa && cd /kaggle/working/villa
git remote add origin https://github.com/ScrollPrize/villa.git
git sparse-checkout set vesuvius volume-cartographer segmentation/models/batchgeneratorsv2 segmentation/models/arch/nnunet
git fetch -q --depth 1 --filter=blob:none origin "$VILLA_SHA"
git checkout -q FETCH_HEAD && git log -1 --format='villa %H %cd'

In [ ]:
%%bash
set -e
pip install -q uv==0.11.24
uv python install "$PY_VER"
cd "$VES" && uv sync --python "$PY_VER" $UV_SYNC_ARGS 2>&1 | tail -5
uv run --no-sync python -c "import torch, sys; print(sys.version.split()[0], torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## 2. Data terms
Already accepted by the owner for this account in exp000. Nothing in this notebook runs `vesuvius.accept_terms`.
If a later cell fails with an error about terms/agreement, **stop and ask the owner** — do not accept on their behalf.

## 3. Checkpoint (pinned HF revision)

In [ ]:
%%bash
set -e
uvx --from huggingface_hub==1.22.0 hf download "$CK_REPO" "$CK_FILE" --revision "$CK_REV" --local-dir /kaggle/working/checkpoints/ink_9um
ls -la "$CKPT" && sha256sum "$CKPT"

## 4. Prepare: 21-slice input from level 0 + ground truth (aborts if the pyramid sanity check fails)

In [ ]:
%%bash
set -eo pipefail
cd "$VES" && uv run --no-sync python /kaggle/working/harness/scripts/exp002_prepare.py --config "$EXP_CFG" --out "$DATA" 2>&1 | grep -v -E "Unclosed|ResponseHandler|TCPConnector|ClientSession|^\s*$" | tail -60

## 5. Inference: 3 depth offsets x 2 directions (layers [2+k, 19+k) of 21)

In [ ]:
%%bash
set -eo pipefail
mkdir -p "$OUT"
cd "$VES"
for k in $OFFSETS; do
  s=$((2 + k)); e=$((19 + k))
  echo "=== offset $k: layers [$s, $e) ==="
  uv run --no-sync python -m "$INFER_MODULE" "$DATA/input.zarr" "$CKPT" "$OUT/pred_off${k}.tif" $INFER_ARGS --layer-start $s --layer-end $e 2>&1 | grep -E "Selected source layer|Selected .* patches|Error|Traceback" 
done
ls -la "$OUT"

## 6. Metrics (numbers only)

In [ ]:
%%bash
set -e
cd "$VES" && uv run --no-sync python /kaggle/working/harness/scripts/exp002_eval.py --config "$EXP_CFG" --data "$DATA" --preds "$OUT" --out "$OUT/metrics" --repo-root /kaggle/working/harness
cp "$DATA/prep.json" "$OUT/metrics/prep.json"
ls -la "$OUT/metrics"

## 7. Bring results back
Download `/kaggle/working/outputs/exp002_841w00_ink9um_s42/metrics/{metrics.json, report.md, prep.json}` (Output/file panel)
and compare `metrics.json`'s sha256 with the one printed above. Do **not** download the prediction TIFFs.